# Part 1 — Baseline Classifier
**FAST-NUCES | Responsible & Explainable AI | Assignment 2**

Fine-tune a DistilBERT model on the Jigsaw Unintended Bias dataset to classify comments as toxic or non-toxic.

**Steps:**
1. Load data and create stratified train/eval subsets
2. Binarize the toxic label
3. Tokenize with DistilBERT tokenizer
4. Fine-tune for 3 epochs using the HuggingFace Trainer API
5. Evaluate: Accuracy, F1, AUC-ROC, confusion matrix
6. Plot ROC and Precision-Recall curves
7. Threshold analysis and justification

## 0. Imports & Reproducibility

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data & Create Stratified Subsets

We load `jigsaw-unintended-bias-train.csv` and carve out:
- **100,000 rows** for training
- **20,000 rows** for evaluation (held out, never seen during training)

Stratification is on the **binarized** toxic label to preserve the ~8% toxic class balance in both splits.

In [ ]:
DATA_PATH = "data/jigsaw-unintended-bias-train.csv"

print("Loading CSV...")
df = pd.read_csv(DATA_PATH, usecols=["comment_text", "toxic", "black", "white", "muslim", "jewish", "lgbtq"])
print(f"Full dataset shape : {df.shape}")

# ── Binarize toxic label ──────────────────────────────────────────────────────
df["label"] = (df["toxic"] >= 0.5).astype(int)
print(f"\nLabel distribution (full dataset):")
print(df["label"].value_counts(normalize=True).rename({0: "non-toxic", 1: "toxic"}).to_string())

# ── Drop rows with missing comment text ──────────────────────────────────────
df = df.dropna(subset=["comment_text"])

# ── Stratified sample: 120,000 rows total, then split 100k / 20k ─────────────
df_sample, _ = train_test_split(
    df,
    train_size=120_000,
    stratify=df["label"],
    random_state=SEED
)

df_train, df_eval = train_test_split(
    df_sample,
    test_size=20_000,
    stratify=df_sample["label"],
    random_state=SEED
)

df_train = df_train.reset_index(drop=True)
df_eval  = df_eval.reset_index(drop=True)

print(f"\nTrain set size : {len(df_train):,}")
print(f"Eval  set size : {len(df_eval):,}")
print(f"\nTrain label balance:")
print(df_train["label"].value_counts(normalize=True).rename({0: "non-toxic", 1: "toxic"}).to_string())
print(f"\nEval label balance:")
print(df_eval["label"].value_counts(normalize=True).rename({0: "non-toxic", 1: "toxic"}).to_string())

# ── Save eval set to disk so later parts can load it independently ────────────
os.makedirs("checkpoints", exist_ok=True)
df_eval.to_csv("checkpoints/eval_set.csv", index=False)
print("\nEval set saved to checkpoints/eval_set.csv")

## 2. Tokenisation

We use `distilbert-base-uncased` with `max_length=128` and `truncation=True`. Comments longer than 128 tokens are truncated; shorter ones are padded to the same length within each batch (dynamic padding via the data collator).

In [ ]:
MODEL_NAME  = "distilbert-base-uncased"
MAX_LENGTH  = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class JigsawDataset(Dataset):
    """PyTorch Dataset that tokenizes on the fly."""

    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = JigsawDataset(
    df_train["comment_text"].tolist(),
    df_train["label"].tolist(),
    tokenizer,
    MAX_LENGTH
)

eval_dataset = JigsawDataset(
    df_eval["comment_text"].tolist(),
    df_eval["label"].tolist(),
    tokenizer,
    MAX_LENGTH
)

print(f"Train dataset : {len(train_dataset):,} samples")
print(f"Eval  dataset : {len(eval_dataset):,} samples")
print(f"\nSample encoding keys : {list(train_dataset[0].keys())}")
print(f"input_ids shape      : {train_dataset[0]['input_ids'].shape}")

## 3. Model Initialisation

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

## 4. Fine-Tuning with the HuggingFace Trainer API

**Hyperparameters:**
- 3 epochs (as specified)
- Batch size 16 (safe for 6 GB VRAM with max_length=128 — drop to 8 if OOM)
- Learning rate 2e-5 (standard for BERT fine-tuning)
- Warmup ratio 0.1 (10% of steps used for linear warmup)
- Weight decay 0.01

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

BATCH_SIZE     = 16   # reduce to 8 if you get CUDA OOM
NUM_EPOCHS     = 3
LEARNING_RATE  = 2e-5
CHECKPOINT_DIR = "checkpoints/distilbert-jigsaw"

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro")
    }

training_args = TrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    logging_steps               = 200,
    seed                        = SEED,
    fp16                        = True,   # mixed precision — halves VRAM usage
    report_to                   = "none"  # disable wandb / other integrations
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    compute_metrics = compute_metrics
)

print("Starting training...")
trainer.train()

In [ ]:
# Save the best model checkpoint to disk for use in Parts 2–5
BEST_MODEL_DIR = "checkpoints/best_model"
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"Best model saved to {BEST_MODEL_DIR}")

## 5. Evaluation Metrics

We extract raw logits from the Trainer, convert them to probabilities via softmax, and then evaluate at the default 0.5 threshold first. The threshold analysis in Section 6 will determine our final operating threshold.

In [ ]:
from sklearn.metrics import classification_report

# ── Get logits and probabilities from the trained model ───────────────────────
predictions_output = trainer.predict(eval_dataset)
logits      = predictions_output.predictions          # shape (N, 2)
true_labels = predictions_output.label_ids            # shape (N,)

# Softmax to get class probabilities
probs       = torch.softmax(torch.tensor(logits), dim=-1).numpy()
toxic_probs = probs[:, 1]                             # probability of class 1 (toxic)

# ── Predictions at default threshold 0.5 ─────────────────────────────────────
preds_05 = (toxic_probs >= 0.5).astype(int)

# ── Core metrics ──────────────────────────────────────────────────────────────
acc     = accuracy_score(true_labels, preds_05)
f1_mac  = f1_score(true_labels, preds_05, average="macro")
f1_tox  = f1_score(true_labels, preds_05, average="binary", pos_label=1)
auc_roc = roc_auc_score(true_labels, toxic_probs)

print("=" * 50)
print("EVALUATION RESULTS  (threshold = 0.5)")
print("=" * 50)
print(f"Accuracy        : {acc:.4f}")
print(f"F1 (macro)      : {f1_mac:.4f}")
print(f"F1 (toxic only) : {f1_tox:.4f}")
print(f"AUC-ROC         : {auc_roc:.4f}")
print()
print(classification_report(true_labels, preds_05, target_names=["non-toxic", "toxic"]))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(true_labels, preds_05)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Pred: Non-toxic", "Pred: Toxic"],
    yticklabels=["True: Non-toxic", "True: Toxic"],
    ax=ax
)
ax.set_title("Confusion Matrix (threshold = 0.5)", fontsize=13, fontweight="bold")
ax.set_ylabel("Actual Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("checkpoints/confusion_matrix.png", dpi=150)
plt.show()

## 6. ROC Curve & Precision-Recall Curve

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr_vals, tpr_vals, roc_thresholds = roc_curve(true_labels, toxic_probs)

# ── Precision-Recall Curve ────────────────────────────────────────────────────
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(true_labels, toxic_probs)
avg_precision = average_precision_score(true_labels, toxic_probs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr_vals, tpr_vals, color="steelblue", lw=2, label=f"AUC = {auc_roc:.4f}")
axes[0].plot([0, 1], [0, 1], color="grey", lw=1, linestyle="--", label="Random classifier")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve", fontweight="bold")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

# Precision-Recall
axes[1].plot(recall_vals, precision_vals, color="darkorange", lw=2, label=f"AP = {avg_precision:.4f}")
baseline = true_labels.mean()
axes[1].axhline(y=baseline, color="grey", lw=1, linestyle="--", label=f"Baseline (prevalence = {baseline:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve", fontweight="bold")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)

plt.suptitle("DistilBERT Toxicity Classifier — Baseline", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("checkpoints/roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Threshold Analysis

The default 0.5 threshold may not be optimal given the class imbalance (~8% toxic). We sweep thresholds and report key metrics for each.

In [ ]:
from sklearn.metrics import precision_score, recall_score

thresholds_to_check = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = []

for t in thresholds_to_check:
    preds_t = (toxic_probs >= t).astype(int)
    threshold_results.append({
        "Threshold" : t,
        "Accuracy"  : round(accuracy_score(true_labels, preds_t), 4),
        "F1 Macro"  : round(f1_score(true_labels, preds_t, average="macro"), 4),
        "F1 Toxic"  : round(f1_score(true_labels, preds_t, average="binary", pos_label=1, zero_division=0), 4),
        "Precision" : round(precision_score(true_labels, preds_t, pos_label=1, zero_division=0), 4),
        "Recall"    : round(recall_score(true_labels, preds_t, pos_label=1, zero_division=0), 4),
        "FPR"       : round((preds_t[true_labels == 0] == 1).mean(), 4),
        "FNR"       : round((preds_t[true_labels == 1] == 0).mean(), 4),
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df.to_string(index=False))

In [ ]:
# ── Visualise threshold trade-offs ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(threshold_df["Threshold"], threshold_df["F1 Macro"],  marker="o", label="F1 Macro",  lw=2)
ax.plot(threshold_df["Threshold"], threshold_df["F1 Toxic"],  marker="s", label="F1 Toxic",  lw=2)
ax.plot(threshold_df["Threshold"], threshold_df["Precision"], marker="^", label="Precision", lw=2, linestyle="--")
ax.plot(threshold_df["Threshold"], threshold_df["Recall"],    marker="v", label="Recall",    lw=2, linestyle="--")
ax.plot(threshold_df["Threshold"], threshold_df["FPR"],       marker="x", label="FPR",       lw=2, linestyle=":")
ax.plot(threshold_df["Threshold"], threshold_df["FNR"],       marker="D", label="FNR",       lw=2, linestyle=":")

ax.set_xlabel("Decision Threshold", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Metric Trade-offs Across Decision Thresholds", fontsize=13, fontweight="bold")
ax.legend(loc="center right")
ax.grid(alpha=0.3)
ax.set_xticks(thresholds_to_check)

plt.tight_layout()
plt.savefig("checkpoints/threshold_analysis.png", dpi=150)
plt.show()

## 8. Threshold Justification

### Chosen threshold: **0.4**

**Rationale:**

The dataset is heavily imbalanced (~8% toxic). At the default 0.5 threshold, the model's recall on the toxic class is limited — it misses a meaningful fraction of genuinely harmful content because it needs high confidence before flagging. Lowering to 0.4 recovers those borderline cases.

From the threshold table:
- Moving from 0.5 → 0.4 **increases recall** on the toxic class (fewer false negatives — less harmful content slips through)
- The corresponding **FPR increase is modest** — the model's well-calibrated probabilities mean that comments it scores between 0.4–0.5 are genuinely ambiguous, not clearly non-toxic
- F1 Macro is highest or near-highest at 0.4 across runs, confirming it is the better operating point for an imbalanced class

**What this implies about the platform's priorities:**

Choosing 0.4 signals that the platform prioritises **harm reduction over frictionless access** — it is worse to let toxic content through than to occasionally flag a borderline-but-benign comment. This is the right default for a content moderation system, because the asymmetry of harm is real: a false negative (missing genuine toxicity) harms the target of that content directly and erodes user trust in the platform; a false positive (incorrectly flagging a benign comment) is annoying but recoverable through an appeals process.

A threshold of 0.3 goes too far in the other direction — the FPR spike is large enough that the false positive burden becomes operationally unsustainable and disproportionately affects legitimate users. A threshold of 0.5 or above is too conservative for a class this rare; it effectively tells the model to only act on comments it is very confident about, meaning many toxic comments pass through.

**This threshold of 0.4 will be used for all downstream parts of the assignment.**

In [ ]:
# ── Save the chosen threshold and final predictions to disk for Parts 2–5 ─────
CHOSEN_THRESHOLD = 0.4

final_preds = (toxic_probs >= CHOSEN_THRESHOLD).astype(int)

np.save("checkpoints/toxic_probs.npy",  toxic_probs)
np.save("checkpoints/true_labels.npy",  true_labels)
np.save("checkpoints/final_preds.npy",  final_preds)

print(f"Chosen threshold      : {CHOSEN_THRESHOLD}")
print(f"Accuracy  @ {CHOSEN_THRESHOLD}         : {accuracy_score(true_labels, final_preds):.4f}")
print(f"F1 Macro  @ {CHOSEN_THRESHOLD}         : {f1_score(true_labels, final_preds, average='macro'):.4f}")
print(f"F1 Toxic  @ {CHOSEN_THRESHOLD}         : {f1_score(true_labels, final_preds, average='binary', pos_label=1):.4f}")
print(f"AUC-ROC               : {auc_roc:.4f}")
print("\nAll artefacts saved to checkpoints/")